# Mutation ↔ Protein Relation-Wise Merge

Merges Mutation–Protein triples from CKG (×2); resolves protein tail names via UniProt;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [32]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MUTATION_PROTEIN/ALL_MUTATION_PROTEIN.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Protein Lookup Dictionary — UniProt

In [2]:
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} entries")

UniProt RecName dict: 794,151 entries


In [24]:
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot
print(f"UniProt AC→Name dict: {len(uniprot_trembl_AC_Name_dict):,} entries")

UniProt AC→Name dict: 252,635,149 entries


## 2. Load KG Sources

### 2a. CKG (×2)

In [9]:
CKG_Mutation_Protein_1 = pd.read_csv(PROC_DIR + 'CKG/File_28_Mutation_Protein_CKG.csv')
CKG_Mutation_Protein_1.columns = CKG_Mutation_Protein_1.columns.str.lower()

CKG_Mutation_Protein_2 = pd.read_csv(PROC_DIR + 'CKG/File_34_Mutation_Protein_CKG.csv')
CKG_Mutation_Protein_2.columns = CKG_Mutation_Protein_2.columns.str.lower()

print(f"CKG 1: {len(CKG_Mutation_Protein_1):,} rows | columns: {list(CKG_Mutation_Protein_1.columns)}")
print(f"CKG 2: {len(CKG_Mutation_Protein_2):,} rows | columns: {list(CKG_Mutation_Protein_2.columns)}")

/tmp/ipykernel_2757454/3553085516.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  CKG_Mutation_Protein_1 = pd.read_csv(PROC_DIR + 'CKG/File_28_Mutation_Protein_CKG.csv')


CKG 1: 132,701,822 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'source', 'kg_source', 'tail_alt_name', 'head_id_is', 'tail_id_is']
CKG 2: 135,598 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'source', 'kg_source', 'tail_alt_name', 'head_id_is', 'tail_id_is', 'effect', 'internal_id', 'evidence', 'interaction']


In [10]:
CKG_Mutation_Protein_1['relation'] = 'Mutation_Protein'
CKG_Mutation_Protein_1['kg_type'] = 'Generalised'
CKG_Mutation_Protein_1['kg_source'] = 'CKG'
CKG_Mutation_Protein_1['species'] = 'Homo species'

CKG_Mutation_Protein_1

,head,relation,tail,head_type,tail_type,source,kg_source,tail_alt_name,head_id_is,tail_id_is,kg_type,species
0,chr6:g.8422602T>A,Mutation_Protein,A0A024QZW4,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
1,chr17:g.30185510G>A,Mutation_Protein,A0A024QZ33,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
2,chr6:g.8429928A>T,Mutation_Protein,A0A024QZW4,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
3,chr17:g.30184979C>G,Mutation_Protein,A0A024QZ33,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
4,chr6:g.8420787T>C,Mutation_Protein,A0A024QZW4,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...
132701817,chr7:g.33015821T>C,Mutation_Protein,X6RM59,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
132701818,chr7:g.33014777C>T,Mutation_Protein,X6RM59,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
132701819,chr7:g.33017559G>T,Mutation_Protein,X6RM59,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species
132701820,chr7:g.33017494C>T,Mutation_Protein,X6RM59,Mutation,Protein,UniProt,CKG,NaN,Mutation,Uniprot_ID,Generalised,Homo species


In [11]:
CKG_Mutation_Protein_2['relation'] = 'Mutation_Protein'

CKG_Mutation_Protein_2['kg_type'] = 'Generalised'
CKG_Mutation_Protein_2['kg_source'] = 'CKG'
CKG_Mutation_Protein_2['species'] = 'Homo species'
CKG_Mutation_Protein_2

,head,relation,tail,head_type,tail_type,source,kg_source,tail_alt_name,head_id_is,tail_id_is,effect,internal_id,evidence,interaction,kg_type,species
0,P49736_p.[Asp80Ala;Tyr81Ala],Mutation_Protein,Q9NVP2,Mutation,Protein,Intact-MutationDs,CKG,Histone chaperone ASF1B {ECO:0000305},Mutation,Uniprot_ID,mutation disrupting strength(MI:1128),EBI-16164801,26167883,"uniprotkb:P49736(protein(MI:0326), 9606 - Homo...",Generalised,Homo species
1,Q14457_p.Phe123Ala,Mutation_Protein,Q16611,Mutation,Protein,Intact-MutationDs,CKG,Bcl-2 homologous antagonist/killer,Mutation,Uniprot_ID,mutation disrupting(MI:0573),EBI-5234926,17446862,"uniprotkb:Q07817-1(protein(MI:0326), 9606 - Ho...",Generalised,Homo species
2,Q9NZ01_p.Pro182Leu,Mutation_Protein,Q9NZ01,Mutation,Protein,Intact-MutationDs,CKG,Very-long-chain enoyl-CoA reductase {ECO:0000305},Mutation,Uniprot_ID,mutation disrupting strength(MI:1128),EBI-24812746,32296183,"uniprotkb:Q9NZ01(protein(MI:0326), 9606 - Homo...",Generalised,Homo species
3,O60656-1_p.Met33Thr,Mutation_Protein,P16662,Mutation,Protein,Intact-MutationDs,CKG,UDP-glucuronosyltransferase 2B7 {ECO:0000303|P...,Mutation,Uniprot_ID,mutation with no effect(MI:2226),EBI-48418883,27857056,"uniprotkb:O60656-1(protein(MI:0326), 9606 - Ho...",Generalised,Homo species
4,Q5S007_p.Gly2019Ser,Mutation_Protein,Q59GY2,Mutation,Protein,Intact-MutationDs,CKG,NaN,Mutation,Uniprot_ID,mutation(MI:0118),EBI-21368474,31046837,"uniprotkb:A8K8U1(protein(MI:0326), 9606 - Homo...",Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135593,Q9H8Y8_p.Met1delinsGlyMet,Mutation_Protein,Q9H8Y8,Mutation,Protein,Intact-MutationDs,CKG,Golgi reassembly-stacking protein 2,Mutation,Uniprot_ID,mutation(MI:0118),EBI-70792362,36115835,"uniprotkb:Q9Y6M7(protein(MI:0326), 9606 - Homo...",Generalised,Homo species
135594,P61019_p.Gln65Leu,Mutation_Protein,Q8R2X8,Mutation,Protein,Intact-MutationDs,CKG,Golgin-45,Mutation,Uniprot_ID,mutation increasing(MI:0382),EBI-4423356,11739401,"uniprotkb:P61019(protein(MI:0326), 9606 - Homo...",Generalised,Homo species
135595,P51532_p.Arg885Cys,Mutation_Protein,Q13620,Mutation,Protein,Intact-MutationDs,CKG,Cullin-4B {ECO:0000305},Mutation,Uniprot_ID,mutation with no effect(MI:2226),EBI-63724217,35512704,"uniprotkb:P51532(protein(MI:0326), 9606 - Homo...",Generalised,Homo species
135596,P13569_p.Phe508del,Mutation_Protein,P09874,Mutation,Protein,Intact-MutationDs,CKG,"Poly [ADP-ribose] polymerase 1, processed N-te...",Mutation,Uniprot_ID,mutation(MI:0118),EBI-27087557,31324722,"uniprotkb:Q9Y5S2(protein(MI:0326), 9606 - Homo...",Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [12]:
SOURCE_DFS = [
    ('CKG_Mutation_Protein_1', CKG_Mutation_Protein_1),
    ('CKG_Mutation_Protein_2', CKG_Mutation_Protein_2),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Mutation_Protein_1] ✓ no duplicates
[CKG_Mutation_Protein_2] ✓ no duplicates


In [13]:
CKG_Mutation_Protein_1 = CKG_Mutation_Protein_1.loc[:, ~CKG_Mutation_Protein_1.columns.duplicated(keep='first')]
CKG_Mutation_Protein_2 = CKG_Mutation_Protein_2.loc[:, ~CKG_Mutation_Protein_2.columns.duplicated(keep='first')]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[CKG_Mutation_Protein_1] ✓ clean
[CKG_Mutation_Protein_2] ✓ clean


## 4. Align Schemas and Concatenate

In [16]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 132,837,420 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,chr6:g.8422602T>A,Mutation_Protein,A0A024QZW4,Mutation,NaN,Protein,CKG,Mutation,Uniprot_ID,NaN,NaN,Generalised,Homo species
1,chr17:g.30185510G>A,Mutation_Protein,A0A024QZ33,Mutation,NaN,Protein,CKG,Mutation,Uniprot_ID,NaN,NaN,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [17]:
final_df['head_type']  = 'Mutation'
final_df['tail_type']  = 'Protein'
final_df['relation']   = 'Mutation_Protein'
final_df['head_id_is'] = 'MutationalVariant'
final_df['tail_id_is'] = 'Uniprot_ID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Mutation_Protein'}
Unique kg_source: {'CKG'}


## 6. Deduplicate

In [18]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 40,864,062 rows


## 7. Resolve Tail (Protein) Names and Add Schema Columns

In [43]:
final_df_dedup['head_detail_name'] = ''
final_df['tail'] = (
    final_df['tail']
    .astype(str)
    .str.split('-')
    .str[0]
)

final_df_dedup['tail_detail_name'] = (
    final_df_dedup['tail_detail_name']
    .fillna(
        final_df_dedup['tail'].map(uniprot_trembl_AC_Name_dict)
        .fillna(
            final_df_dedup['tail'].map(Uniprot_Recname_dict)
        )
    )
)

In [45]:
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()]
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A0B4J1S8_p.Cys64Ala,Mutation_Protein,Q15907,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,Ras-related protein Rab-11B,Generalised,Homo species
1,A0A0B4J1S8_p.Cys64Ala,Mutation_Protein,Q9H3P7,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,"Golgi resident protein GCP60, N-terminally pro...",Generalised,Homo species
2,A0A0B4J1S8_p.Gln61_Lys62delinsAlaAla,Mutation_Protein,Q15907,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,Ras-related protein Rab-11B,Generalised,Homo species
3,A0A0B4J1S8_p.Gln61_Lys62delinsAlaAla,Mutation_Protein,Q9H3P7,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,"Golgi resident protein GCP60, N-terminally pro...",Generalised,Homo species
4,A0A0B4J1S8_p.Gln65_Glu66delinsAlaAla,Mutation_Protein,Q15907,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,Ras-related protein Rab-11B,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40864057,p210 bcr-abl_human_p.[Tyr971Phe;Tyr1016Phe;Tyr...,Mutation_Protein,P29353,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,SHC-transforming protein 1,Generalised,Homo species
40864058,region substitution,Mutation_Protein,O14777,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,Kinetochore protein NDC80 homolog,Generalised,Homo species
40864059,region substitution,Mutation_Protein,Q9H211,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,DNA replication factor Cdt1,Generalised,Homo species
40864060,super_tdu_p.[His5_Phe6delinsAlaAla;Met34Ala;Ph...,Mutation_Protein,Q15561,Mutation,None,Protein,CKG,MutationalVariant,Uniprot_ID,,Transcriptional enhancer factor TEF-3,Generalised,Homo species


## 8. QC — NaN Check

In [46]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,32013911,0,32013911
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 9. Save Output

In [47]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 32,013,911 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MUTATION_PROTEIN/ALL_MUTATION_PROTEIN.parquet
